In [37]:
from dotenv import load_dotenv
import psycopg2
from pgvector.psycopg2 import register_vector
import pandas as pd
from datasets import Dataset

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_core.messages import HumanMessage, SystemMessage



In [3]:
load_dotenv()

True

In [4]:
rate_limiter = InMemoryRateLimiter(
    requests_per_second=7,
    check_every_n_seconds=0.1,
    max_bucket_size=10
)

In [5]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, rate_limiter=rate_limiter)

In [6]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [7]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="rag_db",
    user="postgres"
)
register_vector(conn)
cur = conn.cursor()

In [12]:
def search_chunks(query):
    query_vector = embeddings.embed_query(query)
    search_query = '''
        SELECT chunk_number, chunk_text, images, 1 - (chunk_embedding <=> %s) AS similarity
        FROM chunk_store
        ORDER BY chunk_embedding <=> %s
        LIMIT 5;
    '''
    try:
        cur.execute(search_query, (str(query_vector), str(query_vector)))
        data = cur.fetchall()
        df = pd.DataFrame(data, columns=["chunk_number", "chunk_text", "images", "similarity"])
        return df
    except Exception as e:
        print(f"An error occurred: {e}")
        conn.rollback()

In [56]:
query = "what is the definition of entity data"

In [57]:
df = search_chunks(query)

In [58]:
df

,chunk_number,chunk_text,images,similarity
0,41,Glossary of Terms\n| Endpoint devices ...,[iVBORw0KGgoAAAANSUhEUgAAA8YAAASuCAIAAACP86cvA...,0.458634
1,40,Glossary of Terms\n| Endpoint devices ...,[iVBORw0KGgoAAAANSUhEUgAAA8YAAASuCAIAAACP86cvA...,0.456464
2,39,Glossary of Terms\n| Terms ...,[iVBORw0KGgoAAAANSUhEUgAAA8cAAAS0CAIAAADCfO3sA...,0.436076
3,29,Obfuscation/ masking/ removal of entity attrib...,[],0.402388
4,30,Dataset partitioning\n51 Agencies should reduc...,[],0.379102


In [59]:
def generate_context(df):
    content = []
    for _, row in df.iterrows():
        content.append({
            "type": "text",
            "text": f"--- Chunk {row['chunk_number']} ---\n{row['chunk_text']}"
        })

        if row["images"]:
            for b64_img in row['images']:
                content.append({
                    "type": "image_url",
                    "image_url": {"url": f"data:image/png;base64,{b64_img}"}
                })
    return HumanMessage(content=content)

In [60]:
context = generate_context(df)

In [52]:
def generate_message(query, context):
    messages = [SystemMessage(content="""
    You are a professional assistant. Your goal is to provide accurate answers based ONLY on the provided context chunks.

    RULES:
    1. GROUNDING: If the answer is not contained within the provided context, state clearly that you do not have enough information. Do not use external knowledge.
    2. CITATIONS: You MUST cite the specific Chunk Number (e.g. [Chunk 4]) for every fact or claim you make. Each chunk is labelled at the start (e.g. --- Chunk 4 ---).
    3. CONTEXT INTEGRATION: Treat the chunks as a single unified knowledge base, even if they are provided out of chronological order.
    4. RELEVANCE: Only use information from chunks that are relevant to answering the question.  
    5. TABLES: If context contains Markdown tables, interpret row-to-column relationships strictly to ensure data accuracy.
    6. FORMATTING: Use clear headings and bullet points for complex answers.                          
    """),
    context, 
    HumanMessage(content=f"""
    Question: {query}
    """)

    ]
    return messages

In [61]:
messages = generate_message(query, context)

In [62]:
response = llm.invoke(messages)

In [63]:
print(response.content)

Entity data refers to any data, whether true or not, which are related to an identified or identifiable entity [Chunk 41].


TESTING

In [36]:
test_df = pd.read_csv("ragas_testset.csv")
test_df

,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,What is the intended audience for the Governme...,['Government Data Security Policies | \x001...,The Government Data Security Policies document...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
1,Wht are the key policies in the Govenment's da...,['The Government takes its responsibility as a...,The key policies in the Government's data secu...,Data Security Analyst,MISSPELLED,LONG,single_hop_specific_query_synthesizer
2,What steps should agencies take for effective ...,['Section 1: Data Security Risk Management\n01...,To ensure adequate and effective data security...,Information Security Analyst,WEB_SEARCH_LIKE,MEDIUM,single_hop_specific_query_synthesizer
3,How can agencies minimize the surface area of ...,['Section 3: Reduce the surface area of attack...,Agencies can minimize the surface area of atta...,Information Security Analyst,PERFECT_GRAMMAR,LONG,single_hop_specific_query_synthesizer
4,How do agencies log and monitor data transacti...,['<1-hop>\n\nSection 4: Log and monitor data \...,Agencies log and monitor data transactions to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
5,How do agencies log and monitor data transacti...,['<1-hop>\n\nSection 4: Log and monitor data \...,Agencies log and monitor data transactions to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
6,How do agencies log and monitor data transacti...,['<1-hop>\n\nSection 4: Log and monitor data \...,Agencies log and monitor data transactions to ...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
7,How do logging and monitoring of data transact...,['<1-hop>\n\nSection 4: Log and monitor data \...,Logging and monitoring of data transactions co...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
8,Wht are the key measures in governmnt data sec...,['<1-hop>\n\nGovernment Data Security Policies...,The key measures in government data security p...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
9,Wht r privileged accnts and how do they relate...,['<1-hop>\n\nPrivileged \naccounts\nPrivileged...,Privileged accounts refer to accounts used by ...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer


In [ ]:
evaluation_data = []
for index, row in test_df.iterrows():
    question = row.user_input
    ground_truth = row.reference

    retrieved_df = search_chunks(question)
    context = generate_context(retrieved_df)
    messages = generate_message(question, context)
    response = llm.invoke(messages)

    evaluation_data.append({
        "question": question,
        "answer": response.content,
        "contexts": context_text,
        "ground_truth": ground_truth
    })
eval_dataset = Dataset.from_list(evaluation_data)